# SafeSteer-IN Unified Notebook (Colab + Kaggle)

This notebook supports both Google Colab and Kaggle with a runtime flag.

Pipeline:
1. Setup runtime and paths
2. Prepare dataset and test-file slice filters
3. Extract steering vectors (top-k layers via probe)
4. Run steered evaluation with a fixed alpha on repo-provided test CSV/XLSX
5. Save incremental per-file outputs and final combined CSV

In [ ]:
# Clone repo only if it is not present
import os
from pathlib import Path

if not Path("SafeSteer_IN").exists():
    !git clone https://github.com/MRuhaib/SafeSteer_IN.git
else:
    print("SafeSteer_IN already exists; skipping clone.")

Cloning into 'SafeSteer_IN'...
remote: Enumerating objects: 251, done.
remote: Counting objects: 100% (251/251), done.
remote: Compressing objects: 100% (186/186), done.
remote: Total 251 (delta 61), reused 239 (delta 52), pack-reused 0 (from 0)
Receiving objects: 100% (251/251), 972.13 KiB | 11.05 MiB/s, done.
Resolving deltas: 100% (61/61), done.


In [ ]:
%cd SafeSteer_IN

# Install PyTorch (Colab usually has CUDA already)
#!pip install -r requirements.txt

# Set your HF token
import os
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN", "")
os.environ["SAFESTEER_MODEL"] = os.environ.get("SAFESTEER_MODEL", "sarvam-1")
# Runtime: "auto", "colab", or "kaggle"
RUNTIME = "auto"

if RUNTIME == "auto":
    if Path("/kaggle/working").exists():
        RUNTIME = "kaggle"
    else:
        RUNTIME = "colab"

print("Runtime:", RUNTIME)

# Set your token via env before running, or fill placeholder below

/content/SafeSteer_IN


In [ ]:
# Install dependencies (safe to re-run)
!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt
# SafeSteer does not need torchvision; removing it avoids torch/torchvision CUDA mismatch errors.
!python -m pip uninstall -y torchvision
!python -m pip install --upgrade --force-reinstall transformers==4.49.0 "accelerate>=1.3.0" sentencepiece bitsandbytes openpyxl huggingface_hub
print("Restart the runtime/kernel after this cell before loading models.")

In [ ]:
# ===== USER CONFIG =====
RUNTIME = "auto"  # auto-detect: colab | kaggle
HF_TOKEN = ""     # set if required for gated models

# Dataset source toggle
USE_CLAUDE_DATASET = False
CLAUDE_DATASET_DIR = "data/datasets"
CLAUDE_ALL_PAIRS_FILE = ""

# Used only when USE_CLAUDE_DATASET=False
MIN_PAIRS_PER_SLICE = 20

# Classifier toggle
USE_EXISTING_CLASSIFIER = True
CLASSIFIER_CHECKPOINT_DIR = "models/indic_risk_classifier"
TRAIN_CLASSIFIER_IF_MISSING = False

# Optional training knobs
CLASSIFIER_EPOCHS = 6
CLASSIFIER_BATCH_SIZE = 16

# Vector extraction settings
USE_LAYER_PROBE = True
PROBE_TOP_K = 3  # set 2 or 3 for very fast extraction
PROBE_ONLY_LAYERS = True  # only keep top-k probed layers

# Evaluation settings (expanded repo test set supported: 9 langs x 8 categories x 30 prompts)
USE_EXTERNAL_TEST_DIR = True
TEST_DIR = "data/datasets/expanded"  # repo test CSV/XLSX directory
TEST_FILE_NAME_CONTAINS = []  # optional filename substrings to include; empty => all files
EXTRACT_ONLY_TEST_SLICES = True  # only extract vectors for (language, category) slices found in test files

# Per-model language overrides. [] means auto-detect from selected test files.
SARVAM_LANGUAGES = []
OPENHATHI_LANGUAGES = ["hi"]
KRUTRIM_LANGUAGES = []

# Generation knobs
MAX_PROMPTS_PER_SLICE = 50
MAX_TOKENS = 128

# Optional calibration pre-step (uses scripts/04_calibrate_alpha.py on val split)
RUN_CALIBRATION_FIRST = False
CALIBRATION_ALPHA_VALUES = [12.0]
CALIBRATION_MAX_PAIRS = 20

# Fixed alpha for end-to-end evaluation
RUN_ALPHA_SWEEP = False
ALPHA_VALUES = [12.0]

# Multi-layer steering toggle
USE_MULTILAYER_STEERING = True
MULTILAYER_TOP_K = 3

# Optional model runs
RUN_SARVAM = False
RUN_OPENHATHI = False
RUN_KRUTRIM2 = False
KRUTRIM_QUANTIZE_4BIT = True  # recommended on Kaggle GPUs for 12B model

print("Config loaded.")

Config loaded.


In [ ]:
# ===== PHI CONFIG =====
PHI3_LANGUAGES = []
PHI4_LANGUAGES = []

RUN_PHI3 = False
RUN_PHI4 = False

PHI3_QUANTIZE_4BIT = True  # recommended for Colab/Kaggle
PHI4_QUANTIZE_4BIT = True  # recommended for Colab/Kaggle

print("Phi model config loaded.")

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

def run(cmd: str):
    print(f"\n[RUN] {cmd}")
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    proc = subprocess.Popen(
        cmd,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"Command failed with exit code {rc}: {cmd}")

PROJECT_ROOT = Path.cwd()
print("Project root:", PROJECT_ROOT)
assert (PROJECT_ROOT / "scripts" / "01_build_dataset.py").exists()

if RUNTIME == "kaggle":
    WORKING_OUT = Path("/kaggle/working/safesteer_outputs")
else:
    WORKING_OUT = Path("/content/safesteer_outputs")
WORKING_OUT.mkdir(parents=True, exist_ok=True)
print("Output root:", WORKING_OUT)

SLICE_LANGS = []
SLICE_CATS = []

if USE_EXTERNAL_TEST_DIR:
    import pandas as pd
    test_dir_path = Path(TEST_DIR)
    if not test_dir_path.is_absolute():
        test_dir_path = PROJECT_ROOT / test_dir_path
    test_files = sorted(
        [p for p in test_dir_path.iterdir() if p.suffix.lower() in {".csv", ".xlsx", ".xls"}]
    ) if test_dir_path.exists() else []
    if TEST_FILE_NAME_CONTAINS:
        needles = [s.lower().strip() for s in TEST_FILE_NAME_CONTAINS if str(s).strip()]
        test_files = [p for p in test_files if any(n in p.name.lower() for n in needles)]
    print(f"External test dir: {test_dir_path}")
    print(f"Found {len(test_files)} test files")
    for p in test_files:
        print(" -", p.name)
    assert len(test_files) > 0, "No CSV/XLSX test files found. Update TEST_DIR."

    langs = set()
    cats = set()
    for f in test_files:
        df = pd.read_excel(f) if f.suffix.lower() in {".xlsx", ".xls"} else pd.read_csv(f)
        cols_lower = [str(c).lower() for c in df.columns]
        if all(c.startswith("column") for c in cols_lower) and len(df) > 0:
            first = [str(x).strip() for x in df.iloc[0].tolist()]
            if {"prompt_id", "language", "category", "prompt_text"}.issubset(set(first)):
                df.columns = first
                df = df.iloc[1:].reset_index(drop=True)
        if {"language", "category"}.issubset(set(df.columns)):
            langs.update(str(x).strip() for x in df["language"].dropna().tolist() if str(x).strip())
            cats.update(str(x).strip() for x in df["category"].dropna().tolist() if str(x).strip())

    SLICE_LANGS = sorted(langs)
    SLICE_CATS = sorted(cats)
    print("Detected languages:", SLICE_LANGS)
    print("Detected categories:", SLICE_CATS)

def get_model_slice_args(model_key: str, requested_languages=None):
    """Build --languages/--categories args for this model run."""
    req_langs = [str(x).strip() for x in (requested_languages or []) if str(x).strip()]
    active_langs = sorted(set(req_langs)) if req_langs else list(SLICE_LANGS)
    active_cats = list(SLICE_CATS)

    lang_arg = ""
    cat_arg = ""
    if EXTRACT_ONLY_TEST_SLICES:
        if active_langs:
            lang_arg = " --languages " + " ".join(active_langs)
        if active_cats:
            cat_arg = " --categories " + " ".join(active_cats)

    print(f"[{model_key}] Active languages:", active_langs if active_langs else "all")
    print(f"[{model_key}] Active categories:", active_cats if active_cats else "all")
    return lang_arg, cat_arg, active_langs, active_cats

def export_and_maybe_download(model_key: str):
    """Create per-model zip artifacts immediately after generation and download on Colab."""
    model_root = WORKING_OUT / "model_artifacts" / model_key
    model_root.mkdir(parents=True, exist_ok=True)

    # Copy vectors/results into consolidated output tree
    vec_src = PROJECT_ROOT / "steering_vectors" / model_key
    vec_dst = WORKING_OUT / "steering_vectors" / model_key
    if vec_src.exists():
        vec_dst.parent.mkdir(parents=True, exist_ok=True)
        if vec_dst.exists():
            shutil.rmtree(vec_dst)
        shutil.copytree(vec_src, vec_dst)

    res_src = PROJECT_ROOT / "evaluation" / "results" / "steered_exports" / model_key
    res_dst = WORKING_OUT / "evaluation" / "steered_exports" / model_key
    if res_src.exists():
        res_dst.parent.mkdir(parents=True, exist_ok=True)
        if res_dst.exists():
            shutil.rmtree(res_dst)
        shutil.copytree(res_src, res_dst)

    vec_zip = model_root / f"{model_key}_steering_vectors.zip"
    res_zip = model_root / f"{model_key}_test_responses.zip"

    if vec_src.exists():
        run(f"cd {PROJECT_ROOT} ; zip -r {vec_zip} steering_vectors/{model_key}")
    if res_src.exists():
        run(f"cd {PROJECT_ROOT} ; zip -r {res_zip} evaluation/results/steered_exports/{model_key}")

    print("Created immediate artifacts:")
    print(" -", vec_zip)
    print(" -", res_zip)

    if RUNTIME == "colab":
        try:
            from google.colab import files
            if vec_zip.exists():
                files.download(str(vec_zip))
            if res_zip.exists():
                files.download(str(res_zip))
        except Exception as e:
            print("Colab direct download failed; use file browser to download:", e)
    else:
        print("Kaggle runtime: direct local download is not supported automatically.")
        print("Download these from the Kaggle output/file browser paths above.")

Project root: /content/SafeSteer_IN
Output root: /kaggle/working/safesteer_outputs
External test dir: /content/SafeSteer_IN/data/datasets/test
Found 8 test files
 - anti_minority_sentiment.xlsx
 - caste_discrimination.xlsx
 - child_safety.xlsx
 - code_mixed_toxicity.xlsx
 - communal_religious_hate.xlsx
 - financial_scam.xlsx
 - gender_based_violence.xlsx
 - political_misinformation.xlsx
Detected languages: ['bn', 'gu', 'hi', 'hi-en', 'mr', 'ta']
Detected categories: ['anti_minority_sentiment', 'caste_discrimination', 'child_safety', 'code_mixed_toxicity', 'communal_religious_hate', 'financial_scam', 'gender_based_violence', 'political_misinformation']
Extraction restricted to test-file slices.


In [ ]:
# 1) Prepare dataset
import json
import random

dataset_dir = PROJECT_ROOT / "data" / "datasets"
dataset_dir.mkdir(parents=True, exist_ok=True)

def _read_jsonl(path: Path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def _write_jsonl(path: Path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

if USE_CLAUDE_DATASET:
    src_dir = Path(CLAUDE_DATASET_DIR)
    train_src = src_dir / "train.jsonl"
    val_src = src_dir / "val.jsonl"
    test_src = src_dir / "test.jsonl"

    if train_src.exists():
        shutil.copy2(train_src, dataset_dir / "train.jsonl")
        if val_src.exists() and test_src.exists():
            shutil.copy2(val_src, dataset_dir / "val.jsonl")
            shutil.copy2(test_src, dataset_dir / "test.jsonl")
            print("Copied train/val/test from", src_dir)
        else:
            train_rows = _read_jsonl(dataset_dir / "train.jsonl")
            random.Random(42).shuffle(train_rows)
            n = len(train_rows)
            _write_jsonl(dataset_dir / "train.jsonl", train_rows[: int(0.8 * n)])
            _write_jsonl(dataset_dir / "val.jsonl", train_rows[int(0.8 * n): int(0.9 * n)])
            _write_jsonl(dataset_dir / "test.jsonl", train_rows[int(0.9 * n):])
            print("Auto-split train.jsonl into train/val/test")
    elif CLAUDE_ALL_PAIRS_FILE.strip() and Path(CLAUDE_ALL_PAIRS_FILE).exists():
        all_rows = _read_jsonl(Path(CLAUDE_ALL_PAIRS_FILE))
        random.Random(42).shuffle(all_rows)
        n = len(all_rows)
        _write_jsonl(dataset_dir / "train.jsonl", all_rows[: int(0.8 * n)])
        _write_jsonl(dataset_dir / "val.jsonl", all_rows[int(0.8 * n): int(0.9 * n)])
        _write_jsonl(dataset_dir / "test.jsonl", all_rows[int(0.9 * n):])
        print("Built train/val/test from single JSONL")
    else:
        raise FileNotFoundError("No Claude dataset source found.")
else:
    run("python scripts/01_build_dataset.py --min-pairs-per-slice " + str(MIN_PAIRS_PER_SLICE))

print("Dataset ready at", dataset_dir)

  SafeSteer-IN  ·  Step 1: Dataset Construction
2026-03-22 07:50:30,034  INFO  Built 2160 synthetic pairs (30 per language/category slice)
2026-03-22 07:50:30,041  INFO  Augmented 2160 → 6480 pairs
2026-03-22 07:50:30,045  INFO  After dedup: 6480 pairs
2026-03-22 07:50:30,136  INFO  Wrote /content/SafeSteer_IN/data/datasets/train.jsonl  →  5184 pairs
2026-03-22 07:50:30,148  INFO  Wrote /content/SafeSteer_IN/data/datasets/val.jsonl  →  648 pairs
2026-03-22 07:50:30,159  INFO  Wrote /content/SafeSteer_IN/data/datasets/test.jsonl  →  648 pairs

  Split: train  (5184 pairs)
  Languages: {'te': 582, 'hi-en': 579, 'gu': 567, 'ml': 575, 'kn': 568, 'hi': 562, 'mr': 590, 'ta': 590, 'bn': 571}
  Categories: {'anti_minority_sentiment': 660, 'caste_discrimination': 652, 'child_safety': 660, 'code_mixed_toxicity': 635, 'gender_based_violence': 640, 'financial_scam': 645, 'political_misinformation': 638, 'communal_religious_hate': 654}

  Split: val  (648 pairs)
  Languages: {'kn': 74, 'ta': 62, 'h

In [ ]:
# 2) Reuse classifier checkpoint or train
clf_src = Path(CLASSIFIER_CHECKPOINT_DIR) if CLASSIFIER_CHECKPOINT_DIR else None
clf_dst = PROJECT_ROOT / "models" / "indic_risk_classifier"

if USE_EXISTING_CLASSIFIER:
    if clf_src and clf_src.exists():
        if clf_dst.exists():
            shutil.rmtree(clf_dst)
        shutil.copytree(clf_src, clf_dst)
        print("Copied classifier checkpoint from", clf_src)
    elif clf_dst.exists():
        print("Using existing checkpoint in repo", clf_dst)
    elif TRAIN_CLASSIFIER_IF_MISSING:
        run("python scripts/03_train_classifier.py --epochs " + str(CLASSIFIER_EPOCHS) + " --batch-size " + str(CLASSIFIER_BATCH_SIZE))
    else:
        raise FileNotFoundError("Classifier checkpoint not found.")
else:
    run("python scripts/03_train_classifier.py --epochs " + str(CLASSIFIER_EPOCHS) + " --batch-size " + str(CLASSIFIER_BATCH_SIZE))

clf_out = WORKING_OUT / "models" / "indic_risk_classifier"
if clf_dst.exists():
    if clf_out.exists():
        shutil.rmtree(clf_out)
    shutil.copytree(clf_dst, clf_out)
print("Classifier artifacts copied to", clf_out)

Using classifier checkpoint already present in repo: /content/SafeSteer_IN/models/indic_risk_classifier
Classifier artifacts copied to /kaggle/working/safesteer_outputs/models/indic_risk_classifier


In [ ]:
# 3A/4A) Optional Sarvam-1 run (top-k probed layers)
if RUN_SARVAM:
    sarvam_lang_arg, sarvam_cat_arg, sarvam_active_langs, sarvam_active_cats = get_model_slice_args(
        "sarvam-1", SARVAM_LANGUAGES
    )

    extract_cmd = "python scripts/02_extract_vectors.py --model sarvam-1"
    if USE_LAYER_PROBE:
        extract_cmd += f" --probe-k {PROBE_TOP_K}"
        if PROBE_ONLY_LAYERS:
            extract_cmd += " --probe-only"
    else:
        extract_cmd += " --no-probe"
    extract_cmd += sarvam_lang_arg + sarvam_cat_arg
    run(extract_cmd)

    # Optional: calibrate alpha first (writes calibration_results.json)
    if RUN_CALIBRATION_FIRST:
        cal_cmd = "python scripts/04_calibrate_alpha.py --model sarvam-1"
        if EXTRACT_ONLY_TEST_SLICES and sarvam_active_langs:
            cal_cmd += " --languages " + " ".join(sarvam_active_langs)
        if EXTRACT_ONLY_TEST_SLICES and sarvam_active_cats:
            cal_cmd += " --categories " + " ".join(sarvam_active_cats)
        cal_cmd += " --alphas " + " ".join(str(a) for a in CALIBRATION_ALPHA_VALUES)
        cal_cmd += " --max-pairs " + str(CALIBRATION_MAX_PAIRS)
        run(cal_cmd)

    # Evaluate steered outputs with a fixed alpha
    eval_cmd = (
        "python scripts/05_evaluate.py "
        "--model sarvam-1 --steered-only "
        f"--max-prompts {MAX_PROMPTS_PER_SLICE} --max-tokens {MAX_TOKENS} "
    )
    if USE_EXTERNAL_TEST_DIR:
        eval_cmd += f" --test-dir {TEST_DIR}"
    if EXTRACT_ONLY_TEST_SLICES and sarvam_active_langs:
        eval_cmd += " --languages " + " ".join(sarvam_active_langs)
    if EXTRACT_ONLY_TEST_SLICES and sarvam_active_cats:
        eval_cmd += " --categories " + " ".join(sarvam_active_cats)
    if ALPHA_VALUES:
        eval_cmd += " --alpha-sweep " + " ".join(str(a) for a in ALPHA_VALUES)
    if USE_MULTILAYER_STEERING:
        eval_cmd += " --multi-layer --multi-layer-top-k " + str(MULTILAYER_TOP_K)
    run(eval_cmd)

    # Create/download artifacts immediately for Sarvam
    export_and_maybe_download("sarvam-1")

    print("Sarvam-1 evaluation run complete.")
else:
    print("RUN_SARVAM=False; skipping Sarvam-1 run.")

Streaming output truncated to the last 5000 lines.
                                                                       
2026-03-22 07:52:39,788  INFO    Layer 12  |  norm(raw diff) = 31.5312
2026-03-22 07:52:39,789  INFO    Layer 14  |  norm(raw diff) = 33.3750
2026-03-22 07:52:39,789  INFO    Layer 16  |  norm(raw diff) = 36.5000
2026-03-22 07:52:39,790  INFO    Saved: /content/SafeSteer_IN/steering_vectors/sarvam-1/bn/caste_discrimination/layer12.pt
2026-03-22 07:52:39,790  INFO    Saved: /content/SafeSteer_IN/steering_vectors/sarvam-1/bn/caste_discrimination/layer14.pt
2026-03-22 07:52:39,791  INFO    Saved: /content/SafeSteer_IN/steering_vectors/sarvam-1/bn/caste_discrimination/layer16.pt
2026-03-22 07:52:39,791  INFO    Metadata: /content/SafeSteer_IN/steering_vectors/sarvam-1/bn/caste_discrimination/metadata.json
2026-03-22 07:52:39,791  INFO  Progress: 2/48 slices | extracted=2 skipped=0 | pairs=150 vectors=6
2026-03-22 07:52:39,835  INFO  Loaded 5184 pairs from /content/Safe

In [ ]:
# 3B/4B) Optional OpenHathi run (set OPENHATHI_LANGUAGES=["hi"] for Hindi-only)
if RUN_OPENHATHI:
    oh_lang_arg, oh_cat_arg, oh_active_langs, oh_active_cats = get_model_slice_args(
        "openhathi-base", OPENHATHI_LANGUAGES
)

    extract_cmd = "python scripts/02_extract_vectors.py --model openhathi-base --quantize-4bit"
    if USE_LAYER_PROBE:
        extract_cmd += f" --probe-k {PROBE_TOP_K}"
        if PROBE_ONLY_LAYERS:
            extract_cmd += " --probe-only"
    else:
        extract_cmd += " --no-probe"
    extract_cmd += oh_lang_arg + oh_cat_arg
    run(extract_cmd)

    if RUN_CALIBRATION_FIRST:
        cal_cmd = "python scripts/04_calibrate_alpha.py --model openhathi-base"
        if EXTRACT_ONLY_TEST_SLICES and oh_active_langs:
            cal_cmd += " --languages " + " ".join(oh_active_langs)
        if EXTRACT_ONLY_TEST_SLICES and oh_active_cats:
            cal_cmd += " --categories " + " ".join(oh_active_cats)
        cal_cmd += " --alphas " + " ".join(str(a) for a in CALIBRATION_ALPHA_VALUES)
        cal_cmd += " --max-pairs " + str(CALIBRATION_MAX_PAIRS)
        run(cal_cmd)

    eval_cmd = (
        "python scripts/05_evaluate.py "
        "--model openhathi-base --steered-only "
        f"--max-prompts {MAX_PROMPTS_PER_SLICE} --max-tokens {MAX_TOKENS} "
    )
    if USE_EXTERNAL_TEST_DIR:
        eval_cmd += f" --test-dir {TEST_DIR}"
    if EXTRACT_ONLY_TEST_SLICES and oh_active_langs:
        eval_cmd += " --languages " + " ".join(oh_active_langs)
    if EXTRACT_ONLY_TEST_SLICES and oh_active_cats:
        eval_cmd += " --categories " + " ".join(oh_active_cats)
    if ALPHA_VALUES:
        eval_cmd += " --alpha-sweep " + " ".join(str(a) for a in ALPHA_VALUES)
    if USE_MULTILAYER_STEERING:
        eval_cmd += " --multi-layer --multi-layer-top-k " + str(MULTILAYER_TOP_K)
    run(eval_cmd)

    # Create/download artifacts immediately for OpenHathi
    export_and_maybe_download("openhathi-base")

    print("OpenHathi evaluation run complete.")
else:
    print("RUN_OPENHATHI=False; skipping OpenHathi run.")

In [ ]:
# 3C/4C) Optional Krutrim-2-Instruct run (Kaggle: device_map='auto' can shard across GPUs)
if RUN_KRUTRIM2:
    kr_lang_arg, kr_cat_arg, kr_active_langs, kr_active_cats = get_model_slice_args(
        "krutrim-2-instruct", KRUTRIM_LANGUAGES
)

    extract_cmd = "python scripts/02_extract_vectors.py --model krutrim-2-instruct"
    if KRUTRIM_QUANTIZE_4BIT:
        extract_cmd += " --quantize-4bit"
    if USE_LAYER_PROBE:
        extract_cmd += f" --probe-k {PROBE_TOP_K}"
        if PROBE_ONLY_LAYERS:
            extract_cmd += " --probe-only"
    else:
        extract_cmd += " --no-probe"
    extract_cmd += kr_lang_arg + kr_cat_arg
    run(extract_cmd)

    if RUN_CALIBRATION_FIRST:
        cal_cmd = "python scripts/04_calibrate_alpha.py --model krutrim-2-instruct"
        if EXTRACT_ONLY_TEST_SLICES and kr_active_langs:
            cal_cmd += " --languages " + " ".join(kr_active_langs)
        if EXTRACT_ONLY_TEST_SLICES and kr_active_cats:
            cal_cmd += " --categories " + " ".join(kr_active_cats)
        cal_cmd += " --alphas " + " ".join(str(a) for a in CALIBRATION_ALPHA_VALUES)
        cal_cmd += " --max-pairs " + str(CALIBRATION_MAX_PAIRS)
        run(cal_cmd)

    eval_cmd = (
        "python scripts/05_evaluate.py "
        "--model krutrim-2-instruct --steered-only "
        f"--max-prompts {MAX_PROMPTS_PER_SLICE} --max-tokens {MAX_TOKENS} "
    )
    if USE_EXTERNAL_TEST_DIR:
        eval_cmd += f" --test-dir {TEST_DIR}"
    if EXTRACT_ONLY_TEST_SLICES and kr_active_langs:
        eval_cmd += " --languages " + " ".join(kr_active_langs)
    if EXTRACT_ONLY_TEST_SLICES and kr_active_cats:
        eval_cmd += " --categories " + " ".join(kr_active_cats)
    if ALPHA_VALUES:
        eval_cmd += " --alpha-sweep " + " ".join(str(a) for a in ALPHA_VALUES)
    if USE_MULTILAYER_STEERING:
        eval_cmd += " --multi-layer --multi-layer-top-k " + str(MULTILAYER_TOP_K)
    run(eval_cmd)

    # Create/download artifacts immediately for Krutrim
    export_and_maybe_download("krutrim-2-instruct")

    print("Krutrim-2-Instruct evaluation run complete.")
else:
    print("RUN_KRUTRIM2=False; skipping Krutrim run.")

In [ ]:
# 3D/4D) Optional Phi runs (Colab/Kaggle-friendly multilingual models)
if RUN_PHI3:
    phi3_lang_arg, phi3_cat_arg, phi3_active_langs, phi3_active_cats = get_model_slice_args(
        "phi-3-mini-4k-instruct", PHI3_LANGUAGES
)

    extract_cmd = "python scripts/02_extract_vectors.py --model phi-3-mini-4k-instruct"
    if PHI3_QUANTIZE_4BIT:
        extract_cmd += " --quantize-4bit"
    if USE_LAYER_PROBE:
        extract_cmd += f" --probe-k {PROBE_TOP_K}"
        if PROBE_ONLY_LAYERS:
            extract_cmd += " --probe-only"
    else:
        extract_cmd += " --no-probe"
    extract_cmd += phi3_lang_arg + phi3_cat_arg
    run(extract_cmd)

    if RUN_CALIBRATION_FIRST:
        cal_cmd = "python scripts/04_calibrate_alpha.py --model phi-3-mini-4k-instruct"
        if EXTRACT_ONLY_TEST_SLICES and phi3_active_langs:
            cal_cmd += " --languages " + " ".join(phi3_active_langs)
        if EXTRACT_ONLY_TEST_SLICES and phi3_active_cats:
            cal_cmd += " --categories " + " ".join(phi3_active_cats)
        cal_cmd += " --alphas " + " ".join(str(a) for a in CALIBRATION_ALPHA_VALUES)
        cal_cmd += " --max-pairs " + str(CALIBRATION_MAX_PAIRS)
        run(cal_cmd)

    eval_cmd = (
        "python scripts/05_evaluate.py "
        "--model phi-3-mini-4k-instruct --steered-only "
        f"--max-prompts {MAX_PROMPTS_PER_SLICE} --max-tokens {MAX_TOKENS} "
    )
    if USE_EXTERNAL_TEST_DIR:
        eval_cmd += f" --test-dir {TEST_DIR}"
    if EXTRACT_ONLY_TEST_SLICES and phi3_active_langs:
        eval_cmd += " --languages " + " ".join(phi3_active_langs)
    if EXTRACT_ONLY_TEST_SLICES and phi3_active_cats:
        eval_cmd += " --categories " + " ".join(phi3_active_cats)
    if ALPHA_VALUES:
        eval_cmd += " --alpha-sweep " + " ".join(str(a) for a in ALPHA_VALUES)
    if USE_MULTILAYER_STEERING:
        eval_cmd += " --multi-layer --multi-layer-top-k " + str(MULTILAYER_TOP_K)
    run(eval_cmd)

    export_and_maybe_download("phi-3-mini-4k-instruct")

    print("Phi-3 Mini 4K Instruct evaluation run complete.")
else:
    print("RUN_PHI3=False; skipping Phi-3 run.")

if RUN_PHI4:
    phi4_lang_arg, phi4_cat_arg, phi4_active_langs, phi4_active_cats = get_model_slice_args(
        "phi-4-mini-instruct", PHI4_LANGUAGES
)

    extract_cmd = "python scripts/02_extract_vectors.py --model phi-4-mini-instruct"
    if PHI4_QUANTIZE_4BIT:
        extract_cmd += " --quantize-4bit"
    if USE_LAYER_PROBE:
        extract_cmd += f" --probe-k {PROBE_TOP_K}"
        if PROBE_ONLY_LAYERS:
            extract_cmd += " --probe-only"
    else:
        extract_cmd += " --no-probe"
    extract_cmd += phi4_lang_arg + phi4_cat_arg
    run(extract_cmd)

    if RUN_CALIBRATION_FIRST:
        cal_cmd = "python scripts/04_calibrate_alpha.py --model phi-4-mini-instruct"
        if EXTRACT_ONLY_TEST_SLICES and phi4_active_langs:
            cal_cmd += " --languages " + " ".join(phi4_active_langs)
        if EXTRACT_ONLY_TEST_SLICES and phi4_active_cats:
            cal_cmd += " --categories " + " ".join(phi4_active_cats)
        cal_cmd += " --alphas " + " ".join(str(a) for a in CALIBRATION_ALPHA_VALUES)
        cal_cmd += " --max-pairs " + str(CALIBRATION_MAX_PAIRS)
        run(cal_cmd)

    eval_cmd = (
        "python scripts/05_evaluate.py "
        "--model phi-4-mini-instruct --steered-only "
        f"--max-prompts {MAX_PROMPTS_PER_SLICE} --max-tokens {MAX_TOKENS} "
    )
    if USE_EXTERNAL_TEST_DIR:
        eval_cmd += f" --test-dir {TEST_DIR}"
    if EXTRACT_ONLY_TEST_SLICES and phi4_active_langs:
        eval_cmd += " --languages " + " ".join(phi4_active_langs)
    if EXTRACT_ONLY_TEST_SLICES and phi4_active_cats:
        eval_cmd += " --categories " + " ".join(phi4_active_cats)
    if ALPHA_VALUES:
        eval_cmd += " --alpha-sweep " + " ".join(str(a) for a in ALPHA_VALUES)
    if USE_MULTILAYER_STEERING:
        eval_cmd += " --multi-layer --multi-layer-top-k " + str(MULTILAYER_TOP_K)
    run(eval_cmd)

    export_and_maybe_download("phi-4-mini-instruct")

    print("Phi-4 Mini Instruct evaluation run complete.")
else:
    print("RUN_PHI4=False; skipping Phi-4 run.")

In [ ]:
# Verify outputs and create archives
models_to_check = []
if RUN_SARVAM:
    models_to_check.append("sarvam-1")
if RUN_OPENHATHI:
    models_to_check.append("openhathi-base")
if RUN_KRUTRIM2:
    models_to_check.append("krutrim-2-instruct")
if RUN_PHI3:
    models_to_check.append("phi-3-mini-4k-instruct")
if RUN_PHI4:
    models_to_check.append("phi-4-mini-instruct")

for model_key in models_to_check:
    combined_csv = WORKING_OUT / "evaluation" / "steered_exports" / model_key / "all_categories_steered.csv"
    print(f"{model_key} combined CSV:", combined_csv, "exists=", combined_csv.exists())

    by_file_dir = WORKING_OUT / "evaluation" / "steered_exports" / model_key / "by_test_file"
    if by_file_dir.exists():
        per_file_csvs = sorted(by_file_dir.glob("*_steered.csv"))
        print(f"[{model_key}] Per-test-file CSV count:", len(per_file_csvs))
        for p in per_file_csvs[:10]:
            print(" -", p.name)

if RUNTIME == "kaggle":
    !zip -r /kaggle/working/safesteer_outputs.zip /kaggle/working/safesteer_outputs
else:
    !zip -r /content/safesteer_outputs.zip /content/safesteer_outputs

In [ ]:
# Optional: add, commit, push generated artifacts
GIT_PUSH_RESULTS = False
GIT_COMMIT_MESSAGE = "Add latest steering vectors and steered outputs"

if GIT_PUSH_RESULTS:
    run("git status")
    run("git add steering_vectors evaluation/results data/datasets || true")
    run(f"git commit -m \"{GIT_COMMIT_MESSAGE}\" || echo 'No changes to commit'")
    run("git push origin main")
    print("Push attempted.")
else:
    print("GIT_PUSH_RESULTS=False; skipping git push.")